<a href="https://colab.research.google.com/github/busybee-123/Pollinator_Cam/blob/main/widget_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Interactive Image Review and Sorting (Widget-based)

This section provides an improved interactive way to manually review the consolidated cropped images using `ipywidgets`. You can decide to keep an image, move it to a dedicated `unused bad images` folder, or navigate through them.

In [ ]:
import os
import shutil
from IPython.display import Image, display, clear_output
import ipywidgets as widgets
from ipywidgets import Button, HBox, VBox, Output, Label, IntText, Layout, GridspecLayout
from PIL import Image as PILImage # Use PIL for thumbnail generation
import io # To save PIL image to bytes for ipywidgets.Image

# Define the base project path (parent of 'GBIF images' and 'All Cropped Images')
# `base_gbif_images_path` is expected to be defined in previous cells.
base_project_path = os.path.dirname(base_gbif_images_path)

# Define the path for the consolidated cropped images
all_cropped_images_path = os.path.join(base_project_path, 'All Cropped Images')

# Define the path for bad images
bad_images_path = os.path.join(base_project_path, 'unused bad images')

# Create the bad images folder if it doesn't exist
os.makedirs(bad_images_path, exist_ok=True)
print(f"'All Cropped Images' directory: {all_cropped_images_path}")
print(f"'unused bad images' directory: {bad_images_path}")

# Constants
IMAGES_PER_BATCH = 15 # User requested 20 images
THUMBNAIL_SIZE = (100, 100) # Pixels for thumbnail display

# Function to get initial image list (and refresh it)
def get_image_list():
    images = [f for f in os.listdir(all_cropped_images_path) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif'))]
    images.sort()
    return images

# Global state variables
all_image_filenames = get_image_list()
current_page_start_index = 0 # This will be the start index of the current batch
initial_total_images = len(all_image_filenames)

if not all_image_filenames:
    print(f"No images found in {all_cropped_images_path}. Exiting interactive review.")
else:
    print(f"Found {len(all_image_filenames)} images to review.")

    # Widgets for UI
    image_grid_output = Output() # To display the grid of images
    status_label = Label("") # Will be updated dynamically

    start_image_number_input = IntText(
        value=1,
        min=1,
        max=len(all_image_filenames) if all_image_filenames else 1, # Handle empty list
        step=1,
        description='Start Image #:',
        disabled=False,
        layout=Layout(width='150px')
    )
    goto_image_button = Button(description="Go to Image", layout=Layout(width='100px'))

    next_page_button = Button(description=f"Next {IMAGES_PER_BATCH}")
    prev_page_button = Button(description=f"Previous {IMAGES_PER_BATCH}")
    quit_button = Button(description="Quit")

    # Function to update the displayed image grid and status
    def update_display_grid():
        global current_page_start_index, all_image_filenames

        all_image_filenames = get_image_list() # Re-fetch to get latest state

        if not all_image_filenames:
            with image_grid_output:
                clear_output(wait=True);
                display(widgets.HTML("<h3>No more images to review.</h3>"))
            status_label.value = "Review finished. No images left."
            next_page_button.disabled = True
            prev_page_button.disabled = True
            quit_button.disabled = True
            goto_image_button.disabled = True
            start_image_number_input.disabled = True
            return

        # Adjust current_page_start_index if it goes out of bounds due to deletions
        if current_page_start_index >= len(all_image_filenames):
            current_page_start_index = max(0, len(all_image_filenames) - IMAGES_PER_BATCH)
        if current_page_start_index < 0:
            current_page_start_index = 0

        end_index = min(current_page_start_index + IMAGES_PER_BATCH, len(all_image_filenames))
        current_batch_filenames = all_image_filenames[current_page_start_index:end_index]

        status_label.value = (
            f"Reviewing images {current_page_start_index + 1}-{end_index} of {len(all_image_filenames)} "
            f"(Original total: {initial_total_images})."
        )

        grid_items = []

        for filename in current_batch_filenames:
            full_image_path = os.path.join(all_cropped_images_path, filename)

            if not os.path.exists(full_image_path):
                # This should ideally not happen if get_image_list() is fresh, but good for robustness
                image_widget = widgets.HTML(f"<p>{filename}<br>Not found.</p>")
                mark_bad_btn = Button(description="Missing", disabled=True)
                image_container = VBox([image_widget, mark_bad_btn])
            else:
                try:
                    # Create thumbnail in memory
                    img = PILImage.open(full_image_path)
                    img.thumbnail(THUMBNAIL_SIZE)
                    with io.BytesIO() as f:
                        img.save(f, format='PNG') # Use PNG for widgets, generally better for transparent backgrounds
                        img_bytes = f.getvalue()

                    image_widget = widgets.Image(value=img_bytes, format='png', width=THUMBNAIL_SIZE[0], height=THUMBNAIL_SIZE[1],
                                                 layout=Layout(margin='0px auto')) # Center thumbnail

                    mark_bad_btn = Button(description="Mark Bad", button_style='danger',
                                          layout=Layout(width='auto', margin='5px auto'))
                    # Associate the button with the full path of the image it represents
                    mark_bad_btn.image_path_to_delete = full_image_path
                    mark_bad_btn.image_filename = filename # Store filename for easier handling

                    mark_bad_btn.on_click(on_mark_bad_button_clicked)

                    image_container = VBox([image_widget, Label(filename[:20] + '...' if len(filename) > 20 else filename, layout=Layout(margin='0px auto', font_size='10px')), mark_bad_btn],
                                           layout=Layout(border='1px solid lightgray', padding='5px', margin='2px'))

                except Exception as e:
                    image_widget = widgets.HTML(f"<p>{filename}<br>Error: {e}</p>")
                    mark_bad_btn = Button(description="Error", disabled=True)
                    image_container = VBox([image_widget, mark_bad_btn])

            grid_items.append(image_container)

        # Arrange grid items (e.g., 5 columns, variable rows)
        num_cols = 5
        num_rows = (len(grid_items) + num_cols - 1) // num_cols # Ceiling division
        grid_layout = GridspecLayout(num_rows, num_cols, grid_gap='5px')

        for i, item in enumerate(grid_items):
            row = i // num_cols
            col = i % num_cols
            grid_layout[row, col] = item

        with image_grid_output:
            clear_output(wait=True);
            display(grid_layout)

        # Update button states and start_image_number_input.max
        prev_page_button.disabled = (current_page_start_index == 0)
        next_page_button.disabled = (current_page_start_index + IMAGES_PER_BATCH >= len(all_image_filenames))
        start_image_number_input.max = len(all_image_filenames)


    # Event handlers for navigation
    def on_goto_image_button_clicked(b):
        global current_page_start_index
        requested_image_num = start_image_number_input.value
        if 1 <= requested_image_num <= len(all_image_filenames):
            # Calculate the starting index of the batch that contains the requested image
            current_page_start_index = ((requested_image_num - 1) // IMAGES_PER_BATCH) * IMAGES_PER_BATCH
            update_display_grid()
        else:
            with image_grid_output:
                print(f"Invalid image number. Please enter a number between 1 and {len(all_image_filenames)}.")

    def on_next_page_button_clicked(b):
        global current_page_start_index
        if current_page_start_index + IMAGES_PER_BATCH < len(all_image_filenames):
            current_page_start_index += IMAGES_PER_BATCH
            update_display_grid()
        else:
            with image_grid_output:
                print("Already at the last page.")

    def on_prev_page_button_clicked(b):
        global current_page_start_index
        if current_page_start_index > 0:
            current_page_start_index = max(0, current_page_start_index - IMAGES_PER_BATCH)
            update_display_grid()
        else:
            with image_grid_output:
                print("Already at the first page.")

    def on_mark_bad_button_clicked(b):
        global all_image_filenames
        image_to_delete_path = b.image_path_to_delete
        image_to_delete_filename = b.image_filename
        dest_path = os.path.join(bad_images_path, image_to_delete_filename)

        with image_grid_output:
            try:
                shutil.move(image_to_delete_path, dest_path)
                print(f"Moved {image_to_delete_filename} to '{os.path.basename(bad_images_path)}'.")
                # The list will be refreshed in update_display_grid, so no explicit pop needed here
                update_display_grid() # Refresh the grid to remove the marked bad image
            except Exception as e:
                print(f"Error moving {image_to_delete_filename}: {e}")

    def on_quit_button_clicked(b):
        with image_grid_output:
            clear_output(wait=True);
            print("Exiting interactive review. Review finished.")
        status_label.value = "Interactive review ended by user."
        next_page_button.disabled = True
        prev_page_button.disabled = True
        quit_button.disabled = True
        goto_image_button.disabled = True
        start_image_number_input.disabled = True


    # Attach handlers
    goto_image_button.on_click(on_goto_image_button_clicked)
    next_page_button.on_click(on_next_page_button_clicked)
    prev_page_button.on_click(on_prev_page_button_clicked)
    quit_button.on_click(on_quit_button_clicked)

    # Display UI
    display(VBox([
        status_label,
        HBox([start_image_number_input, goto_image_button]),
        HBox([prev_page_button, next_page_button, quit_button]),
        image_grid_output
    ]))

    # Initial display
    update_display_grid()

    print("\n--- Interactive review setup. Use the buttons above. ---")